# Global, step and branch metrics

Where a metric attaches decides what it sees:

| attach with | sees | on records |
|---|---|---|
| no `nodes` (global) | the flow's output (`approved`, `payment`) | all |
| `nodes="loans/rate"` (a step) | what that step wrote (`rate` before pricing) | all that ran it |
| `nodes="loans/pricing/unsecured"` (a step in a branch arm) | that arm's `rate` | only those the arm took |
| `nodes="loans/pricing/is_secured"` (the branch's condition) | `is_secured` | all |
| `nodes="loans/pricing"` (the branch) | what the branch passes on (`rate` after pricing) | all |
| `nodes="loans/pricing/*"` (a glob) | every step under the branch | each step's own |

The scenario raises the unsecured premium, so only the unsecured arm's rates move.

In [1]:
from decider.experiments import *
from example_projects.experiments.loan_pipeline import applications, build

ENTRY = "example_projects.experiments.loan_pipeline:build"
pipeline = build()   # ratio -> rate -> pricing (secured | unsecured) -> approved, payment
loans = applications(200_000)
loans.head()

income,debt,collateral
f64,f64,f64
3829.0,718.0,1344.0
3454.0,938.0,3913.0
4704.0,1956.0,0.0
3797.0,1960.0,16821.0
2939.0,387.0,0.0


In [2]:
RATE = dict(column="rate", range=(0.05, 0.35), bins=30)
result = (Experiment("where-metrics-attach", pipeline, loans, entry=ENTRY)
          .scenario("premium_8pct", params={"loans": {"pricing": {"unsecured": {"premium": 0.08}}}})
          .metric(
              # global
              Divergence(column="approved"),
              Describe(column="payment"),
              # a step
              Histogram(name="rate_before_pricing", nodes="loans/rate", **RATE),
              # a step inside a branch arm, and the branch's condition
              Histogram(name="rate_unsecured_arm", nodes="loans/pricing/unsecured", **RATE),
              ValueCounts(name="secured", column="is_secured", nodes="loans/pricing/is_secured"),
              # the branch itself, and records' paths through it
              Histogram(name="rate_after_pricing", nodes="loans/pricing", **RATE),
              BranchFlows(outcome="approved"),
          )
          .plot(Distribution(metric="rate_before_pricing"), Distribution(metric="rate_unsecured_arm"),
                Distribution(metric="rate_after_pricing"), Ranges(metric="describe"),
                Sankey(name="flows_baseline", metric="branch_flows"),
                Sankey(name="flows_premium_8pct", metric="branch_flows", scenario="premium_8pct"))
          .run())
result.plots["rate_unsecured_arm"].show()
result.plots["rate_after_pricing"].show()

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.05, 'end': 0.060000000000000005, 'records': 0}, {'scenario': 'baseline', 'start': 0.060000000000000005, 'end': 0.07, 'records': 0}, {'scenario': 'baseline', 'start': 0.07, 'end': 0.08, 'records': 0}, {'scenario': 'baseline', 'start': 0.08, 'end': 0.09, 'records': 0}, {'scenario': 'baseline', 'start': 0.09, 'end': 0.1, 'records': 0}, {'scenario': 'baseline', 'start': 0.1, 'end': 0.11, 'records': 0}, {'scenario': 'baseline', 'start': 0.11, 'end': 0.12000000000000001, 'records': 0}, {'scenario': 'baseline', 'start': 0.12000000000000001, 'end': 0.13, 'records': 0}, {'scenario': 'baseline', 'start': 0.13, 'end': 0.14, 'records': 0}, {'scenario': 'baseline', 'start': 0.14, 'end': 0.15000000000000002, 'records': 10841}, {'scenario': 'baseline', 'start': 0.15000000000000002, 'end': 0.16, 'records': 21059}, {'scenario': 'baseline', 'start': 0.16, 'end': 0.16999999999999998, 'records': 19866}, {'scenario': 'baseline', 'start': 0.16999999999999998, 'end': 0.18, 'records': 14607}, {'scenario': 'baseline', 'start': 0.18, 'end': 0.19, 'records': 8368}, {'scenario': 'baseline', 'start': 0.19, 'end': 0.2, 'records': 3814}, {'scenario': 'baseline', 'start': 0.2, 'end': 0.21000000000000002, 'records': 1215}, {'scenario': 'baseline', 'start': 0.21000000000000002, 'end': 0.22000000000000003, 'records': 211}, {'scenario': 'baseline', 'start': 0.22000000000000003, 'end': 0.22999999999999998, 'records': 16}, {'scenario': 'baseline', 'start': 0.22999999999999998, 'end': 0.24, 'records': 0}, {'scenario': 'baseline', 'start': 0.24, 'end': 0.25, 'records': 0}, {'scenario': 'baseline', 'start': 0.25, 'end': 0.26, 'records': 0}, {'scenario': 'baseline', 'start': 0.26, 'end': 0.27, 'records': 0}, {'scenario': 'baseline', 'start': 0.27, 'end': 0.28, 'records': 0}, {'scenario': 'baseline', 'start': 0.28, 'end': 0.29, 'records': 0}, {'scenario': 'baseline', 'start': 0.29, 'end': 0.3, 'records': 0}, {'scenario': 'baseline', 'start': 0.3, 'end': 0.31, 'records': 0}, {'scenario': 'baseline', 'start': 0.31, 'end': 0.32, 'records': 0}, {'scenario': 'baseline', 'start': 0.32, 'end': 0.33, 'records': 0}, {'scenario': 'baseline', 'start': 0.33, 'end': 0.33999999999999997, 'records': 0}, {'scenario': 'baseline', 'start': 0.33999999999999997, 'end': 0.35, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.05, 'end': 0.060000000000000005, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.060000000000000005, 'end': 0.07, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.07, 'end': 0.08, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.08, 'end': 0.09, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.09, 'end': 0.1, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.1, 'end': 0.11, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.11, 'end': 0.12000000000000001, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.12000000000000001, 'end': 0.13, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.13, 'end': 0.14, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.14, 'end': 0.15000000000000002, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.15000000000000002, 'end': 0.16, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.16, 'end': 0.16999999999999998, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.16999999999999998, 'end': 0.18, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.18, 'end': 0.19, 'records': 5669}, {'scenario': 'premium_8pct', 'start': 0.19, 'end': 0.2, 'records': 12901}, {'scenario': 'premium_8pct', 'start': 0.2, 'end': 0.21000000000000002, 'records': 15024}, {'scenario': 'premium_8pct', 'start': 0.21000000000000002, 'end': 0.22000000000000003, 'records': 13957}, {'scenario': 'premium_8pct', 'start': 0.22000000000000003, 'end': 0.22999999999999998, 'records': 11718}, {'scenario': 'premium_8pct', 'start': 0.22999999999999998, 'end

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.05, 'end': 0.060000000000000005, 'records': 0}, {'scenario': 'baseline', 'start': 0.060000000000000005, 'end': 0.07, 'records': 0}, {'scenario': 'baseline', 'start': 0.07, 'end': 0.08, 'records': 0}, {'scenario': 'baseline', 'start': 0.08, 'end': 0.09, 'records': 41194}, {'scenario': 'baseline', 'start': 0.09, 'end': 0.1, 'records': 50577}, {'scenario': 'baseline', 'start': 0.1, 'end': 0.11, 'records': 23193}, {'scenario': 'baseline', 'start': 0.11, 'end': 0.12000000000000001, 'records': 4833}, {'scenario': 'baseline', 'start': 0.12000000000000001, 'end': 0.13, 'records': 206}, {'scenario': 'baseline', 'start': 0.13, 'end': 0.14, 'records': 0}, {'scenario': 'baseline', 'start': 0.14, 'end': 0.15000000000000002, 'records': 10841}, {'scenario': 'baseline', 'start': 0.15000000000000002, 'end': 0.16, 'records': 21059}, {'scenario': 'baseline', 'start': 0.16, 'end': 0.16999999999999998, 'records': 19866}, {'scenario': 'baseline', 'start': 0.16999999999999998, 'end': 0.18, 'records': 14607}, {'scenario': 'baseline', 'start': 0.18, 'end': 0.19, 'records': 8368}, {'scenario': 'baseline', 'start': 0.19, 'end': 0.2, 'records': 3814}, {'scenario': 'baseline', 'start': 0.2, 'end': 0.21000000000000002, 'records': 1215}, {'scenario': 'baseline', 'start': 0.21000000000000002, 'end': 0.22000000000000003, 'records': 211}, {'scenario': 'baseline', 'start': 0.22000000000000003, 'end': 0.22999999999999998, 'records': 16}, {'scenario': 'baseline', 'start': 0.22999999999999998, 'end': 0.24, 'records': 0}, {'scenario': 'baseline', 'start': 0.24, 'end': 0.25, 'records': 0}, {'scenario': 'baseline', 'start': 0.25, 'end': 0.26, 'records': 0}, {'scenario': 'baseline', 'start': 0.26, 'end': 0.27, 'records': 0}, {'scenario': 'baseline', 'start': 0.27, 'end': 0.28, 'records': 0}, {'scenario': 'baseline', 'start': 0.28, 'end': 0.29, 'records': 0}, {'scenario': 'baseline', 'start': 0.29, 'end': 0.3, 'records': 0}, {'scenario': 'baseline', 'start': 0.3, 'end': 0.31, 'records': 0}, {'scenario': 'baseline', 'start': 0.31, 'end': 0.32, 'records': 0}, {'scenario': 'baseline', 'start': 0.32, 'end': 0.33, 'records': 0}, {'scenario': 'baseline', 'start': 0.33, 'end': 0.33999999999999997, 'records': 0}, {'scenario': 'baseline', 'start': 0.33999999999999997, 'end': 0.35, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.05, 'end': 0.060000000000000005, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.060000000000000005, 'end': 0.07, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.07, 'end': 0.08, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.08, 'end': 0.09, 'records': 41194}, {'scenario': 'premium_8pct', 'start': 0.09, 'end': 0.1, 'records': 50577}, {'scenario': 'premium_8pct', 'start': 0.1, 'end': 0.11, 'records': 23193}, {'scenario': 'premium_8pct', 'start': 0.11, 'end': 0.12000000000000001, 'records': 4833}, {'scenario': 'premium_8pct', 'start': 0.12000000000000001, 'end': 0.13, 'records': 206}, {'scenario': 'premium_8pct', 'start': 0.13, 'end': 0.14, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.14, 'end': 0.15000000000000002, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.15000000000000002, 'end': 0.16, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.16, 'end': 0.16999999999999998, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.16999999999999998, 'end': 0.18, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.18, 'end': 0.19, 'records': 5669}, {'scenario': 'premium_8pct', 'start': 0.19, 'end': 0.2, 'records': 12901}, {'scenario': 'premium_8pct', 'start': 0.2, 'end': 0.21000000000000002, 'records': 15024}, {'scenario': 'premium_8pct', 'start': 0.21000000000000002, 'end': 0.22000000000000003, 'records': 13957}, {'scenario': 'premium_8pct', 'start': 0.22000000000000003, 'end': 0.22999999999999998, 'records': 11718}, {'scenario': 'premium_8pct', 

Before pricing the two runs agree; inside the unsecured arm the whole distribution shifts; after the
branch only the unsecured share of records moved. `show()` puts every metric's table next to its charts.

In [3]:
result.show()

### where-metrics-attach: completed

200,000 records · revision `b7091562ef82` · 1 scenario(s) next to the baseline

scenario,status,records,revision,changes,error
str,str,i64,null,str,null
"""baseline""","""completed""",200000,null,"""unchanged""",null
"""premium_8pct""","""completed""",200000,null,"""params {""loans"": {""pricing"": {…",null


#### divergence

`divergence` of `approved` in the flow's output

scenario,column,compared,changed,first,samples
str,str,i64,i64,i64,str
"""baseline""","""approved""",200000,0,null,"""[]"""
"""premium_8pct""","""approved""",200000,42720,14,"""[{""row"": 14, ""expected"": true,…"


Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'changed': 0, 'compared': 200000, 'first': None}, {'scenario': 'premium_8pct', 'changed': 42720, 'compared': 200000, 'first': 14}]}, 'mark': {'type': 'bar'}, 'encoding': {'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'changed', 'type': 'quantitative'}, {'field': 'compared', 'type': 'quantitative'}, {'field': 'first', 'title': 'first changed record', 'type': 'quantitative'}], 'x': {'field': 'scenario', 'sort': None, 'type': 'nominal'}, 'y': {'field': 'changed', 'title': 'records changed', 'type': 'quantitative'}}, 'title': 'approved changed vs baseline', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})

...

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.05, 'end': 0.060000000000000005, 'records': 0}, {'scenario': 'baseline', 'start': 0.060000000000000005, 'end': 0.07, 'records': 0}, {'scenario': 'baseline', 'start': 0.07, 'end': 0.08, 'records': 0}, {'scenario': 'baseline', 'start': 0.08, 'end': 0.09, 'records': 41194}, {'scenario': 'baseline', 'start': 0.09, 'end': 0.1, 'records': 50577}, {'scenario': 'baseline', 'start': 0.1, 'end': 0.11, 'records': 23193}, {'scenario': 'baseline', 'start': 0.11, 'end': 0.12000000000000001, 'records': 4833}, {'scenario': 'baseline', 'start': 0.12000000000000001, 'end': 0.13, 'records': 206}, {'scenario': 'baseline', 'start': 0.13, 'end': 0.14, 'records': 0}, {'scenario': 'baseline', 'start': 0.14, 'end': 0.15000000000000002, 'records': 10841}, {'scenario': 'baseline', 'start': 0.15000000000000002, 'end': 0.16, 'records': 21059}, {'scenario': 'baseline', 'start': 0.16, 'end': 0.16999999999999998, 'records': 19866}, {'scenario': 'baseline', 'start': 0.16999999999999998, 'end': 0.18, 'records': 14607}, {'scenario': 'baseline', 'start': 0.18, 'end': 0.19, 'records': 8368}, {'scenario': 'baseline', 'start': 0.19, 'end': 0.2, 'records': 3814}, {'scenario': 'baseline', 'start': 0.2, 'end': 0.21000000000000002, 'records': 1215}, {'scenario': 'baseline', 'start': 0.21000000000000002, 'end': 0.22000000000000003, 'records': 211}, {'scenario': 'baseline', 'start': 0.22000000000000003, 'end': 0.22999999999999998, 'records': 16}, {'scenario': 'baseline', 'start': 0.22999999999999998, 'end': 0.24, 'records': 0}, {'scenario': 'baseline', 'start': 0.24, 'end': 0.25, 'records': 0}, {'scenario': 'baseline', 'start': 0.25, 'end': 0.26, 'records': 0}, {'scenario': 'baseline', 'start': 0.26, 'end': 0.27, 'records': 0}, {'scenario': 'baseline', 'start': 0.27, 'end': 0.28, 'records': 0}, {'scenario': 'baseline', 'start': 0.28, 'end': 0.29, 'records': 0}, {'scenario': 'baseline', 'start': 0.29, 'end': 0.3, 'records': 0}, {'scenario': 'baseline', 'start': 0.3, 'end': 0.31, 'records': 0}, {'scenario': 'baseline', 'start': 0.31, 'end': 0.32, 'records': 0}, {'scenario': 'baseline', 'start': 0.32, 'end': 0.33, 'records': 0}, {'scenario': 'baseline', 'start': 0.33, 'end': 0.33999999999999997, 'records': 0}, {'scenario': 'baseline', 'start': 0.33999999999999997, 'end': 0.35, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.05, 'end': 0.060000000000000005, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.060000000000000005, 'end': 0.07, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.07, 'end': 0.08, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.08, 'end': 0.09, 'records': 41194}, {'scenario': 'premium_8pct', 'start': 0.09, 'end': 0.1, 'records': 50577}, {'scenario': 'premium_8pct', 'start': 0.1, 'end': 0.11, 'records': 23193}, {'scenario': 'premium_8pct', 'start': 0.11, 'end': 0.12000000000000001, 'records': 4833}, {'scenario': 'premium_8pct', 'start': 0.12000000000000001, 'end': 0.13, 'records': 206}, {'scenario': 'premium_8pct', 'start': 0.13, 'end': 0.14, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.14, 'end': 0.15000000000000002, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.15000000000000002, 'end': 0.16, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.16, 'end': 0.16999999999999998, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.16999999999999998, 'end': 0.18, 'records': 0}, {'scenario': 'premium_8pct', 'start': 0.18, 'end': 0.19, 'records': 5669}, {'scenario': 'premium_8pct', 'start': 0.19, 'end': 0.2, 'records': 12901}, {'scenario': 'premium_8pct', 'start': 0.2, 'end': 0.21000000000000002, 'records': 15024}, {'scenario': 'premium_8pct', 'start': 0.21000000000000002, 'end': 0.22000000000000003, 'records': 13957}, {'scenario': 'premium_8pct', 'start': 0.22000000000000003, 'end': 0.22999999999999998, 'records': 11718}, {'scenario': 'premium_8pct', 

#### branch_flows

`branch_flows` of `approved` at steps matching `*`

scenario,outcome,nodes,links
str,str,str,str
"""baseline""","""approved""","""[{""id"": ""in"", ""label"": ""200000…","""[{""source"": ""in"", ""target"": ""l…"
"""premium_8pct""","""approved""","""[{""id"": ""in"", ""label"": ""200000…","""[{""source"": ""in"", ""target"": ""l…"


<Vega chart: Record flow through branch arms into an outcome. Adapted from David Bacci's Sankey chart (https://github.com/PBI-David/Deneb-Showcase, MIT License, Copyright (c) 2023 Davide Bacci); see sankey.vg.LICENSE.>

<Vega chart: Record flow through branch arms into an outcome. Adapted from David Bacci's Sankey chart (https://github.com/PBI-David/Deneb-Showcase, MIT License, Copyright (c) 2023 Davide Bacci); see sankey.vg.LICENSE.>